# N040 · 堆排序与非比较排序

**目标：** 依据键类型和值域选择排序机制。

**先修：** N037, N039, N011。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 堆结构导览；计数/桶/基数；稳定子过程；比较下界适用条件。

**配套讲解来源：** [同名逐章意见](../../comment/040_heapsort_counting_bucket_radix.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决的问题是：**按"键的类型和取值范围"挑排序工具**。比较排序（插入、快排、归并）只靠两两比较，任何这种排序最坏都要 Ω(n log n)；但如果键是不大的整数，我们可以绕开比较，直接用"数格子"的办法把时间压到接近线性。本章一手教**堆排序**（比较排序里唯一 O(1) 额外空间又稳在 O(n log n) 的），一手教两种非比较排序：**计数排序**和**基数排序**。

先看三个具体到数字的例子（均来自 notebook 真实数据）：

- **计数排序**：输入 `[3,1,3]` → 输出 `[1,3,3]`。做法是数出"值 1 出现 1 次、值 3 出现 2 次"，再按值从小到大倒出来——完全没做任何两两比较。
- **基数排序（“运行示例”部分 的数据）**：输入 `[170,45,75,90,802,24,2,66]` → 输出 `[2,24,45,66,75,90,170,802]`。做法是按"个位 → 十位 → 百位"三轮稳定分桶，每轮只看一位数字（详见第三节手算）。
- **堆排序**：输入 `[4,1,3,2]` → 输出 `[1,2,3,4]`。先原地把它整理成最大堆 `[4,2,3,1]`，再反复"取堆顶放到末尾、修复剩余堆"。

还有一个反向例子同样重要：`counting_sort([0,10**9])` 会抛 `ValueError`——两个值把值域撑到十亿宽，数格子要开十亿个格子的数组，宁可报错也不硬来。“思路推导”部分 的警告"禁止对巨大稀疏值域无条件分配数组"就是指这个。

## 2. 基础知识：先读懂这些词

- **堆（heap）**：一棵父节点永远不小于子节点的完全二叉树（最大堆）。注意它只管"父亲压得住儿子"，**不要求整棵树从左到右有序**——这是堆和"排好序的数组"的本质区别。
- **数组藏树**：不用真的建树，把节点按层放进数组：下标 i 的孩子是 2i+1 和 2i+2，父亲是 (i-1)//2。代码里的 `2*root+1` 就是在数组里找左孩子。
- **下沉（sift down）**：如果某个节点压不住孩子，就拿它和"较大的那个孩子"交换并继续往下比，直到压得住或没有孩子为止。这是修复堆的基本动作。
- **建堆（build heap）**：从最后一个非叶节点（下标 len(a)//2-1）倒着回到根，逐个下沉。倒序处理保证下沉时孩子的子树已经是堆。
- **堆排序（heapsort）**：建最大堆后反复做三件事——堆顶是当前最大值，把它换到数组末尾，把堆的规模减一，再从根下沉修复。
- **计数排序（counting sort）**：值都在一段不大范围内时，开一个计数数组数每个值出现几次，再把计数展开成有序输出。复杂度 O(n + R)，R 是值域宽度。
- **桶排序（bucket sort）**：把值域切成若干桶，元素按值落桶、桶内再各自排序（常用插入排序）。它是计数排序的"一格变一桶"推广。
- **基数排序（radix sort，LSD 低位优先）**：对整数从最低位数字到最高位，逐位做一轮**稳定**的分桶收集。所有轮次做完，整体有序。
- **稳定性在非比较排序中的地位**：计数排序倒序回填、基数排序逐位分桶，都靠"稳定"才能把低位已经排好的次序保持到高位。稳定子过程不是锦上添花，是正确性的一部分。
- **比较下界的适用条件**：Ω(n log n) 下界只约束"只靠两两比较"的排序；计数、基数不走比较路线，所以能突破它——代价是必须知道值域、且值域可控。

## 3. 思路推导：从小例子开始

### 思路一：堆排序

- **Step 1：建最大堆。** 从最后一个非叶节点到根，逐个下沉。建完后 a[0] 是全数组最大值。
- **Step 2：取顶归位。** 交换 a[0] 与当前堆的最后一个元素——最大值落到了它最终的位置（数组末尾）。
- **Step 3：缩小堆并修复。** 堆规模减一，根可能不再压得住孩子，从根下沉一次。
- **Step 4：重复 Step 2–3**，直到堆里只剩一个元素，整个数组从右到左依次归位，有序。

**手算演示：`heap_sort([4,1,3,2])`。** 建堆阶段（对下标 1、0 依次下沉）：sift(1)：节点 1 的值 1 压不住孩子 2，交换得 [4,2,3,1]；sift(0)：4 压得住 3，不动。堆建成 `[4,2,3,1]`。取顶阶段：end=3：交换 a[0],a[3] 得 [1,2,3,4]，对规模 3 从根下沉，1 与较大孩子 3 交换得 [3,2,1,4]；end=2：交换 a[0],a[2] 得 [1,2,3,4]，1 与孩子 2 交换得 [2,1,3,4]；end=1：交换 a[0],a[1] 得 `[1,2,3,4]`，堆只剩一个元素，收工。

### 思路二：计数排序

- **Step 1：测值域。** 求最小值 low 和最大值 high，宽度 width = high - low + 1；宽度超限直接报错。
- **Step 2：计数。** 每个元素给 counts[键 - low] 加一（减 low 是把最小值平移到 0 号格子，这样负数也能用同一套格子）。
- **Step 3：前缀和。** counts[i] 累加后变成"键 ≤ i+low 的元素总个数"，也就是该键最后一个元素应放的位置（开）。
- **Step 4：倒序回填。** 从数组**末尾**往前放元素：先给 counts[键] 减一，再把元素放进 out[减一后的位置]。倒序 + 先减一是稳定的关键：同键元素中，越靠后的放进越后的格子。

**手算演示：`counting_sort([3,1,3])`。** low=1，width=3。计数得 counts=[1,0,2]（值 1 一个、值 3 两个）。前缀和得 [1,1,3]。倒序回填：x=3 → counts[2] 减为 2，out[2]=3；x=3 → counts[2] 减为 1，out[1]=3；x=1 → counts[0] 减为 0，out[0]=1。输出 `[1,3,3]`。全程 0 次元素比较。

### 思路三：基数排序（低位优先）

- **Step 1：按当前位分桶。** 用 `x//place%10` 取出当前位（place 为 1、10、100…），把元素追加进 0–9 号桶。
- **Step 2：按桶号顺序收集。** 从 0 号桶到 9 号桶依序倒回数组。桶内追加是尾接，收集保持同桶元素的相对次序——这一步必须稳定。
- **Step 3：位权乘 10，重复**，直到位权超过最大值，所有位都处理完。

**手算演示：“运行示例”部分 的 `a=[170,45,75,90,802,24,2,66]`（正是 show_table 展示的三行）。**

| 当前位权 | 分桶依据 | 稳定分桶后 |
|---|---|---|
| 1（个位） | 0,5,5,0,2,4,2,6 | [170, 90, 802, 2, 24, 45, 75, 66] |
| 10（十位） | 7,9,0,0,2,4,7,6 | [802, 2, 24, 45, 66, 170, 75, 90] |
| 100（百位） | 8,0,0,0,0,1,0,0 | [2, 24, 45, 66, 75, 90, 170, 802] |

第一轮按个位：个位为 0 的 170、90 先进 0 号桶（保持原序），个位为 2 的 802、2 进 2 号桶……收集得第二列。第二轮按十位：802 和 2 的十位都是 0，稳定分桶让 802 仍在 2 前面（它们个位已排好 2 < 802？不——个位轮里 2 在 802 后面吗？回看第一轮：个位 2 号桶内顺序是 802、2，因为原数组里 802 在 2 前面、追加保持了这个先后；而 2 < 802，说明此时同十位内次序"暂时不对"），第三轮按百位：百位为 0 的桶收集时按第二列顺序收进 2、24、45、66、75、90——个位、十位已经排好的次序被稳定地保留下来，最终整体有序。三轮之后 `place=1000 > 802`，循环结束，输出 `[2,24,45,66,75,90,170,802]`。

## 4. 核心代码：heap_sort

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def heap_sort(a):

    def sift(root, size):
        while 2 * root + 1 < size:
            child = 2 * root + 1
            if child + 1 < size and a[child + 1] > a[child]:
                child += 1
            if a[root] >= a[child]:
                break
            a[root], a[child] = (a[child], a[root])
            root = child
    for i in range(len(a) // 2 - 1, -1, -1):
        sift(i, len(a))
    for end in range(len(a) - 1, 0, -1):
        a[0], a[end] = (a[end], a[0])
        sift(0, end)
    return a
```

### 逐段读懂代码

### 函数 1：`heap_sort` —— 堆排序

```python
def heap_sort(a):

    def sift(root, size):
        while 2 * root + 1 < size:
            child = 2 * root + 1
            if child + 1 < size and a[child + 1] > a[child]:
                child += 1
            if a[root] >= a[child]:
                break
            a[root], a[child] = (a[child], a[root])
            root = child
    for i in range(len(a) // 2 - 1, -1, -1):
        sift(i, len(a))
    for end in range(len(a) - 1, 0, -1):
        a[0], a[end] = (a[end], a[0])
        sift(0, end)
    return a
```

- `def sift(root,size):`：内嵌的下沉函数。root 是待修复子树的根，size 是"当前堆"的有效长度（堆只占 a[:size]，size 之后是已归位的排序区）。
- `while 2*root+1<size:`：只要 root 还有左孩子（左孩子下标在堆内）就继续。
- `if child+1<size and a[child+1]>a[child]: child+=1`：挑两个手里**较大**的那个孩子（有右孩子且右孩子更大才切过去）。和较大的孩子换，才能保证换上来的新父亲也压得住另一个孩子。
- `if a[root]>=a[child]: break`：父亲已经不小于孩子，堆性质已满足，提前收工——这就是"部分有序就不再动"的剪枝。
- `a[root],a[child]=a[child],a[root]; root=child`：交换后旧根落到孩子位置，继续以该位置为新根往下审。
- `for i in range(len(a)//2-1,-1,-1): sift(i,len(a))`：建堆。len(a)//2-1 是最后一个非叶节点（再往后的元素都是叶子，天然是堆），倒序逐个下沉，保证处理每个节点时它的子树已是堆。
- `for end in range(len(a)-1,0,-1):`：取顶归位循环。end 是当前堆的规模，从 n-1 缩到 1。
- `a[0],a[end]=a[end],a[0]; sift(0,end)`：堆顶（最大值）换到 a[end] 归位；堆规模缩到 end，从根下沉修复。全程只用 O(1) 个变量，原地完成。

### 函数 2：`counting_sort` —— 计数排序

```python
def counting_sort(a, key=lambda x: x, max_range=1000000):
    if not a:
        return []
    low = min(map(key, a))
    high = max(map(key, a))
    width = high - low + 1
    if width > max_range:
        raise ValueError('value range too large for counting sort')
    counts = [0] * width
    for x in a:
        counts[key(x) - low] += 1
    for i in range(1, width):
        counts[i] += counts[i - 1]
    out = [None] * len(a)
    for x in reversed(a):
        i = key(x) - low
        counts[i] -= 1
        out[counts[i]] = x
    return out
```

- `key=lambda x:x, max_range=1000000`：key 让它能排"按某属性"（测试里对元组按首字段排）；max_range 是值域宽度上限，超过就报错，兑现"禁止对巨大稀疏值域无条件分配数组"。
- `if not a: return []`：空输入直接返回空列表，省得后面 min/max 对空序列抛错。
- `low=min(map(key,a)); ...; width=high-low+1`：测值域；`key(x)-low` 把键平移到 0..width-1，负数键也能用（练习 1 的关键）。
- `for x in a: counts[key(x)-low]+=1`：数格子，每个键的出现次数加一。
- `for i in range(1,width): counts[i]+=counts[i-1]`：前缀和，counts[i] 变成"键小于等于该格的元素总数"，即该键元素能放到的最右位置（开区间）。
- `for x in reversed(a):`：**必须倒序**回填。同键元素中，原数组靠后的应该放更后；倒序遍历让它最先被处理、配到最右的空位（先减一再用），于是同键元素的相对次序原样保留——这就是计数排序稳定的原因。
- `return out`：返回新列表，不改原数组。

### 函数 3：`radix_sort_nonnegative` —— 基数排序

```python
def radix_sort_nonnegative(a):
    if any((x < 0 for x in a)):
        raise ValueError('nonnegative integers required')
    out = list(a)
    place = 1
    maximum = max(out, default=0)
    while place <= maximum:
        buckets = [[] for _ in range(10)]
        for x in out:
            buckets[x // place % 10].append(x)
        out = [x for bucket in buckets for x in bucket]
        place *= 10
    return out
```

- `if any(x<0 for x in a): raise ValueError(...)`：入口守卫。负数的"低位"没有无歧义的十进制表示，本章版只收非负整数，遇到负数明确报错。
- `out=list(a)`：复制一份，不碰调用方的原数组。
- `place=1; maximum=max(out,default=0)`：place 是当前位权（1、10、100…）；`default=0` 让空输入的 max 返回 0，于是 while 条件 1<=0 不成立，空输入直接返回空列表。
- `while place<=maximum:`：只要当前位权还没超过最大值，就还有没处理的位。最多循环 d 次，d 是最大值的十进制位数（802 → 3 轮）。
- `buckets=[[] for _ in range(10)]`：十个新桶。注意必须用列表推导生成十个**独立**的列表；写成 `[[]]*10` 会得到十个指向同一个列表的引用，全进一个桶。
- `for x in out: buckets[x//place%10].append(x)`：`x//place%10` 一步取出当前位数字（place=10 时 45//10%10=4）。追加进桶保持同桶元素的相对次序——稳定。
- `out=[x for bucket in buckets for x in bucket]`：按桶号 0→9 收集回一维列表，双层推导的读法是"对每个桶、依序取桶里每个元素"。
- `return out`：所有位处理完，整体有序。

## 5. 分段实现：按顺序运行

**本章接口：** `heap_sort(a)`、`counting_sort(a)`、`radix_sort_nonnegative(a)`

### heap_sort

In [1]:
def heap_sort(a):

    def sift(root, size):
        while 2 * root + 1 < size:
            child = 2 * root + 1
            if child + 1 < size and a[child + 1] > a[child]:
                child += 1
            if a[root] >= a[child]:
                break
            a[root], a[child] = (a[child], a[root])
            root = child
    for i in range(len(a) // 2 - 1, -1, -1):
        sift(i, len(a))
    for end in range(len(a) - 1, 0, -1):
        a[0], a[end] = (a[end], a[0])
        sift(0, end)
    return a

### counting_sort

In [2]:
def counting_sort(a, key=lambda x: x, max_range=1000000):
    if not a:
        return []
    low = min(map(key, a))
    high = max(map(key, a))
    width = high - low + 1
    if width > max_range:
        raise ValueError('value range too large for counting sort')
    counts = [0] * width
    for x in a:
        counts[key(x) - low] += 1
    for i in range(1, width):
        counts[i] += counts[i - 1]
    out = [None] * len(a)
    for x in reversed(a):
        i = key(x) - low
        counts[i] -= 1
        out[counts[i]] = x
    return out

### radix_sort_nonnegative

In [3]:
def radix_sort_nonnegative(a):
    if any((x < 0 for x in a)):
        raise ValueError('nonnegative integers required')
    out = list(a)
    place = 1
    maximum = max(out, default=0)
    while place <= maximum:
        buckets = [[] for _ in range(10)]
        for x in out:
            buckets[x // place % 10].append(x)
        out = [x for bucket in buckets for x in bucket]
        place *= 10
    return out

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[170,45,75,90,802,24,2,66]; rows=[]; out=a[:]
for place in [1,10,100]:
    buckets=[[] for _ in range(10)]
    for x in out: buckets[x//place%10].append(x)
    out=[x for b in buckets for x in b]; rows.append((place,out[:]))
show_table(['当前位权','稳定分桶后'],rows)

当前位权,稳定分桶后
1,"[170, 90, 802, 2, 24, 45, 75, 66]"
10,"[802, 2, 24, 45, 66, 170, 75, 90]"
100,"[2, 24, 45, 66, 75, 90, 170, 802]"


## 7. 正确性、适用条件与复杂度

堆顶为剩余最大值，逐个放入已排序后缀保持归纳关系。基数分桶稳定，使新位相同的元素保持低位已排好的次序，逐位归纳得到整体有序。

**代价：** 堆 O(n log n)时间/O(1)辅助空间；计数 O(n+R)时间空间；d位十进制基数 O(d(n+10))时间/O(n+10)空间。

### 展开理解

**堆排序为什么对？** 每一轮取顶之前，堆顶都是"当前堆里所有元素"的最大值（堆性质一层层往上推：每个父亲不小于孩子，根就不小于任何人）。把它换到 a[end] 后，它右边的元素都已经在各自最终位置上（每轮归位一个，且都 ≥ 后来归位的），堆规模缩一格后再从根下沉一次，剩余部分重新满足堆性质。重复 n-1 轮，从右到左每个位置都放上了"剩余中的最大值"，整个数组自然非降。建堆那一遍为什么合法？我们是从后往前逐个下沉的，处理节点 i 时它的两棵子树已经是堆，下沉只是把 i 接进去，不破坏子树的堆性质。

**计数排序为什么对？** 前缀和之后，counts[i] 的含义是"键 ≤ 第 i 格对应键的元素个数"，也就是这个键的元素在输出中能占据的最右边界。回填时每放一个就减一，同键元素的格子从右往左被精确占满，不多不少——每个元素都拿到唯一格子，格子又按键升序排列，所以输出有序且是原数组的一个重排。倒序回填保证稳定性，第四节已细说。

**基数排序为什么对？** 用一句话归纳：做完第 k 轮后，数组按"后 k 位组成的数"有序。第一轮后按个位有序。假设做完第 k-1 轮按后 k-1 位有序，第 k 轮做**稳定**分桶：高位数字小的桶先收集，所以先看高位就能分出大小；高位相同的元素落在同一个桶里，稳定分桶保持它们进入时的次序，而那个次序正是"按后 k-1 位排好的"次序。位权超过最大值时所有位都算完，整体有序。如果分桶不稳，这个论证立刻塌掉——这就是"稳定子过程"的地位。

**复杂度是多少？** 堆排序：建堆 O(n)，取顶 n-1 轮每轮下沉 O(log n)，共 O(n log n)、额外空间 O(1)。计数排序：一遍计数 O(n)、一遍前缀和 O(R)、一遍回填 O(n)，共 O(n+R)、空间 O(n+R)。基数排序：d 轮、每轮分桶收集 O(n+10)，共 O(d(n+10))。拿数字感受：n = 10⁵ 个 0..99999 的整数，计数排序约 n+R ≈ 2×10⁵ 次数组操作，比任何比较排序的 170 万次还快一个量级；但同样的 n 如果值域是 0..10⁹，R 太大，计数排序直接出局，此时堆排序照跑，基数排序 d=10 轮约 10×10⁵ 也可以。选型一句话：**键是不大的整数找非比较排序，否则老老实实比较排序**。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言逐条看：

- 随机块（`for n in range(50): a=[rng.randrange(1000) ...]`）：**交叉验证测试（三个实现互为裁判）**。长度 0 到 49、值 0..999 的随机数组，断言 `heap_sort(a[:])==counting_sort(a)==radix_sort_nonnegative(a)==sorted(a)`——三种排序的输出与内置 sorted 三方一致。注意 `heap_sort(a[:])` 复制了输入（heap 原地），而 counting/radix 本身不改输入。
- `assert counting_sort(a,key=lambda x:x[0])==[(1,'b'),(2,'a'),(2,'c')]`（a=[(2,'a'),(1,'b'),(2,'c')]）：**稳定性测试**。按首字段排，键同为 2 的 (2,'a') 与 (2,'c') 必须保持 'a' 在前的原序。这条专抓"回填没有倒序"的 bug——正序回填会让 (2,'c') 抢到靠前的格子。
- `for fn,values in [(radix_sort_nonnegative,[1,-1]),(counting_sort,[0,10**9])]:` 错误块：**契约测试（不支持的域必须显式失败）**。radix 遇到 -1 必须抛 ValueError（负数超出教学契约）；counting 遇到 [0,10**9] 值域宽度十亿、超过 max_range 也必须抛 ValueError。`try/except` 的写法是"抛了 ValueError 算通过（pass），没抛反而主动 raise AssertionError"——强制失败要失败得响亮，不许悄悄算。

In [5]:
from random import Random

rng = Random(40)

for n in range(50):
    a = [rng.randrange(1000) for _ in range(n)]
    assert heap_sort(a[:]) == counting_sort(a) == radix_sort_nonnegative(a) == sorted(a)

a = [(2, 'a'), (1, 'b'), (2, 'c')]

assert counting_sort(a, key=lambda x: x[0]) == [(1, 'b'), (2, 'a'), (2, 'c')]

for fn, values in [(radix_sort_nonnegative, [1, -1]), (counting_sort, [0, 10 ** 9])]:
    try:
        fn(values)
    except ValueError:
        pass
    else:
        raise AssertionError('unsupported domain must fail explicitly')

print('N040: 所有本章断言通过')

N040: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 支持有负数的计数排序。

**练习 2：** 解释桶排序平均线性需要什么假设。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（支持有负数的计数排序）**：方向提示——难点只在"数组下标不能是负数"。两种路子：其一，减去最小值做平移（`key(x)-low`），这正是本章代码已经采用的办法，你要做的是论证它对负数同样成立并用 [-3,-1,-2] 手算一遍；其二，把负数非负数拆成两段分别排再拼接（注意负数段要倒序）。写清边界：全负数组、-0 邻近值、以及"值域过宽仍然必须报错"的保留条款（[0,10**9] 依旧抛错，负数支持和值域限制是两回事）。
- **练习 2（解释桶排序平均线性需要什么假设）**：方向提示——桶排序把值域均匀切成约 n 个桶，期望每个桶只有 O(1) 个元素，桶内插入排序期望 O(1)，总计 O(n)。这个"期望"的前提是**元素在值域上分布大致均匀**：如果 10⁵ 个数全挤进同一个桶，桶内插入排序退回 O(n²)（或桶内改用归并则 O(n log n)），线性就不成立了。可以设计两个对照输入：均匀分布的 [0, 10⁶) 随机数 vs 全部集中在 [0, 100) 的随机数，手算或实测桶内最大长度，用数字说明假设何时破产。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
def heap_sort(a):

    def sift(root, size):
        while 2 * root + 1 < size:
            child = 2 * root + 1
            if child + 1 < size and a[child + 1] > a[child]:
                child += 1
            if a[root] >= a[child]:
                break
            a[root], a[child] = (a[child], a[root])
            root = child
    for i in range(len(a) // 2 - 1, -1, -1):
        sift(i, len(a))
    for end in range(len(a) - 1, 0, -1):
        a[0], a[end] = (a[end], a[0])
        sift(0, end)
    return a

def counting_sort(a, key=lambda x: x, max_range=1000000):
    if not a:
        return []
    low = min(map(key, a))
    high = max(map(key, a))
    width = high - low + 1
    if width > max_range:
        raise ValueError('value range too large for counting sort')
    counts = [0] * width
    for x in a:
        counts[key(x) - low] += 1
    for i in range(1, width):
        counts[i] += counts[i - 1]
    out = [None] * len(a)
    for x in reversed(a):
        i = key(x) - low
        counts[i] -= 1
        out[counts[i]] = x
    return out

def radix_sort_nonnegative(a):
    if any((x < 0 for x in a)):
        raise ValueError('nonnegative integers required')
    out = list(a)
    place = 1
    maximum = max(out, default=0)
    while place <= maximum:
        buckets = [[] for _ in range(10)]
        for x in out:
            buckets[x // place % 10].append(x)
        out = [x for bucket in buckets for x in bucket]
        place *= 10
    return out

# 示例与回归测试
from random import Random

rng = Random(40)

for n in range(50):
    a = [rng.randrange(1000) for _ in range(n)]
    assert heap_sort(a[:]) == counting_sort(a) == radix_sort_nonnegative(a) == sorted(a)

a = [(2, 'a'), (1, 'b'), (2, 'c')]

assert counting_sort(a, key=lambda x: x[0]) == [(1, 'b'), (2, 'a'), (2, 'c')]

for fn, values in [(radix_sort_nonnegative, [1, -1]), (counting_sort, [0, 10 ** 9])]:
    try:
        fn(values)
    except ValueError:
        pass
    else:
        raise AssertionError('unsupported domain must fail explicitly')

print('N040: 所有本章断言通过')

N040: 所有本章断言通过


## 11. 代表题与迁移

“题目映射”部分 列出的题目：

- **912. Sort an Array（排序数组）**：这题练的是排序选型——本章的 heap_sort（O(n log n)、O(1) 空间、无递归）和 radix_sort_nonnegative（整数、O(d(n+10))）都能过，正好验证"按键类型选机制"的主题。
- **164. Maximum Gap（最大间距）**：排序后相邻元素的最大差值。这题练的是非比较排序的思维方式：利用鸽笼原理按值域分桶，n 个数落在 [min,max] 上，最大间距至少是 (max-min)/(n-1)，于是桶内不可能产生答案、只比桶间空隙即可，O(n) 出解——与计数/桶排序"用值域结构换时间"是同一招。

照 notebook 原文提醒：以上条目用于知识映射，本章实现遵循教学契约，刷题时以 LeetCode 官方题面与评测为准。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。